# Growing Neural Cellular Automata

A tiny neural network learns a *local update rule*. Apply it to every pixel
of a grid, over and over, and a target image grows itself from a single
seed pixel -- and if you damage it mid-growth, it heals.

Based on: Mordvintsev, Randazzo, Niklasson, Levin, **"Growing Neural
Cellular Automata"**, Distill, 2020. https://distill.pub/2020/growing-ca/

**Before running:** Runtime -> Change runtime type -> GPU (T4 is plenty).
On a T4 this trains ~15-25x faster than on a laptop CPU -- the full
8000-iteration run should take roughly 10-20 minutes instead of several hours.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

## 1. Target image
Draws a star with a transparent background. Swap this for any RGBA PNG you like (just make sure the background is transparent).

In [ ]:
def make_target(size=40):
    hi_res = size * 4
    img = Image.new("RGBA", (hi_res, hi_res), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)
    cx, cy = hi_res / 2, hi_res / 2
    r_outer = hi_res * 0.42
    r_inner = r_outer * 0.42
    points = []
    for i in range(10):
        angle = -np.pi / 2 + i * np.pi / 5
        r = r_outer if i % 2 == 0 else r_inner
        points.append((cx + r * np.cos(angle), cy + r * np.sin(angle)))
    draw.polygon(points, fill=(255, 140, 40, 255))
    inner_points = [(cx + 0.55 * (x - cx), cy + 0.55 * (y - cy)) for x, y in points]
    draw.polygon(inner_points, fill=(255, 210, 90, 255))
    img = img.resize((size, size), Image.LANCZOS)
    return np.array(img).astype(np.float32) / 255.0

GRID_SIZE = 40   # bump up from 32 -> 40 now that we have a GPU to spare
target_np = make_target(GRID_SIZE)
plt.imshow(target_np); plt.axis("off"); plt.title("target"); plt.show()

## 2. The model
Fixed Sobel perception (no learned weights) + a tiny learned 2-layer update MLP (as 1x1 convs), applied identically to every cell.

In [ ]:
CHANNEL_N = 16
CELL_FIRE_RATE = 0.5

def make_seed(size, channel_n=CHANNEL_N, n=1, device=DEVICE):
    seed = torch.zeros(n, channel_n, size, size, device=device)
    seed[:, 3:, size // 2, size // 2] = 1.0
    return seed

def get_living_mask(x):
    alpha = x[:, 3:4, :, :]
    return F.max_pool2d(alpha, kernel_size=3, stride=1, padding=1) > 0.1

class CAModel(nn.Module):
    def __init__(self, channel_n=CHANNEL_N, hidden_n=128):
        super().__init__()
        self.channel_n = channel_n
        identity = torch.zeros(3, 3); identity[1, 1] = 1.0
        sobel_x = torch.tensor([[-1.,0.,1.],[-2.,0.,2.],[-1.,0.,1.]]) / 8.0
        sobel_y = sobel_x.t()
        kernel = torch.stack([identity, sobel_x, sobel_y], dim=0).unsqueeze(1)
        kernel = kernel.repeat(channel_n, 1, 1, 1)
        self.register_buffer("perception_kernel", kernel)
        self.update_net = nn.Sequential(
            nn.Conv2d(channel_n * 3, hidden_n, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(hidden_n, channel_n, kernel_size=1, bias=False),
        )
        nn.init.zeros_(self.update_net[-1].weight)

    def perceive(self, x):
        return F.conv2d(x, self.perception_kernel, padding=1, groups=self.channel_n)

    def forward(self, x, fire_rate=CELL_FIRE_RATE):
        pre_life_mask = get_living_mask(x)
        y = self.perceive(x)
        dx = self.update_net(y)
        update_mask = (torch.rand(x[:, :1, :, :].shape, device=x.device) <= fire_rate).float()
        x = x + dx * update_mask
        post_life_mask = get_living_mask(x)
        life_mask = (pre_life_mask & post_life_mask).float()
        return x * life_mask

def to_rgb(x):
    rgb, a = x[:, :3, :, :], torch.clamp(x[:, 3:4, :, :], 0, 1)
    return 1.0 - a + rgb

## 3. Training
Sample pool (learns persistence, not just one-shot growth) + random damage (this single ingredient is what makes regeneration emerge -- it's never rewarded directly).

In [ ]:
POOL_SIZE = 256
BATCH_SIZE = 8
N_ITERS = 8000          # the paper's setting -- fine on a GPU, too slow on CPU
LR = 2e-3

def make_circle_mask(size, device):
    x = torch.arange(size, device=device).float()
    yy, xx = torch.meshgrid(x, x, indexing="ij")
    cx, cy = np.random.uniform(0.2, 0.8) * size, np.random.uniform(0.2, 0.8) * size
    r = np.random.uniform(0.1, 0.25) * size
    return ((xx - cx) ** 2 + (yy - cy) ** 2 > r ** 2).float()

target = torch.tensor(target_np).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
model = CAModel(channel_n=CHANNEL_N).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[int(N_ITERS*0.7)], gamma=0.1)

seed = make_seed(GRID_SIZE, device=DEVICE)
pool = seed.repeat(POOL_SIZE, 1, 1, 1).clone()
loss_log = []

for it in range(N_ITERS):
    idx = np.random.choice(POOL_SIZE, BATCH_SIZE, replace=False)
    x = pool[idx].clone()
    with torch.no_grad():
        batch_losses = ((x[:, :4] - target) ** 2).mean(dim=[1, 2, 3])
        worst = batch_losses.argmax().item()
        x[worst] = seed[0]
        for i in range(BATCH_SIZE):
            if i != worst and np.random.rand() < 0.5:
                x[i] = x[i] * make_circle_mask(GRID_SIZE, DEVICE)

    n_steps = np.random.randint(64, 96)
    for _ in range(n_steps):
        x = model(x)

    loss = F.mse_loss(x[:, :4], target.expand(BATCH_SIZE, -1, -1, -1))
    opt.zero_grad(); loss.backward()
    for p in model.parameters():
        if p.grad is not None:
            p.grad = p.grad / (p.grad.norm() + 1e-8)
    opt.step(); sched.step()

    pool[idx] = x.detach()
    loss_log.append(loss.item())
    if it % 100 == 0:
        print(f"iter {it:5d}  loss {loss.item():.5f}")

torch.save(model.state_dict(), "nca_weights.pt")
plt.plot(loss_log); plt.yscale("log"); plt.title("training loss"); plt.show()

## 4. Growth + regeneration demo

In [ ]:
@torch.no_grad()
def tensor_to_np(x):
    img = to_rgb(x)[0].detach().cpu().permute(1, 2, 0).clamp(0, 1).numpy()
    return img

@torch.no_grad()
def show_growth(model, n_steps=150, n_snaps=8):
    x = make_seed(GRID_SIZE, device=DEVICE)
    snaps_at = set(np.linspace(0, n_steps - 1, n_snaps).astype(int))
    fig, axes = plt.subplots(1, n_snaps, figsize=(2 * n_snaps, 2))
    snap_i = 0
    for step in range(n_steps):
        if step in snaps_at:
            axes[snap_i].imshow(tensor_to_np(x)); axes[snap_i].axis("off"); snap_i += 1
        x = model(x)
    plt.show()

@torch.no_grad()
def show_regeneration(model, grow_steps=120, heal_steps=150, n_snaps=6):
    x = make_seed(GRID_SIZE, device=DEVICE)
    for _ in range(grow_steps):
        x = model(x)
    mask = torch.ones(1, 1, GRID_SIZE, GRID_SIZE, device=DEVICE)
    yy, xx = torch.meshgrid(torch.arange(GRID_SIZE, device=DEVICE).float(),
                             torch.arange(GRID_SIZE, device=DEVICE).float(), indexing="ij")
    cx = cy = GRID_SIZE / 2; r = GRID_SIZE * 0.22
    mask[0, 0] = ((xx - cx) ** 2 + (yy - cy) ** 2 > r ** 2).float()
    x = x * mask
    snaps_at = set(np.linspace(0, heal_steps - 1, n_snaps).astype(int))
    fig, axes = plt.subplots(1, n_snaps + 1, figsize=(2 * (n_snaps + 1), 2))
    axes[0].imshow(tensor_to_np(x)); axes[0].axis("off"); axes[0].set_title("damaged")
    snap_i = 1
    for step in range(heal_steps):
        x = model(x)
        if step in snaps_at:
            axes[snap_i].imshow(tensor_to_np(x)); axes[snap_i].axis("off"); snap_i += 1
    plt.show()

show_growth(model)
show_regeneration(model)

## Where to go next

- **Localize the response**: PCA the hidden channels down to 3 and render as
  false color to visualize the "signal" propagating from a wound.
- **Multiple patterns, one model**: condition the update net on a one-hot
  "which pattern" vector so a single trained network can grow several
  different images.
- **3D**: extend the grid to voxels, the perception filters to 3D Sobel
  kernels, and grow a simple voxel shape instead of a 2D image.
- **Interactive browser demo**: export `nca_weights.pt` and re-implement this
  same forward pass (it's just two convolutions) in JavaScript/WebGL so
  anyone can click to damage it live with zero install.
- **Robustness ablation**: vary damage size/shape/timing during training and
  measure how regeneration quality changes -- turns this into an actual
  research-style study for a final report.